# MEIKURAL Version 2 — raw audio inference

Run from the repository root or `notebooks/`, using the project virtual environment. Install dependencies and train a checkpoint first; see `README.md` and `docs/ver2.tex`.

Input is **audio only**, of any positive duration. Sa is estimated once from the entire supplied recording and reused for every window. The CNN and STFT settings are unchanged from Version 1. The model ranks seven supported raags; scores are comparative, not calibrated probabilities. Popular songs may mix raags or be outside these classes. Overlapping windows are correlated.

This notebook contains no training or feature-extraction implementation; it imports the same pipeline used by the commands.

**Before use:** expand the Version 2 datasets, prepare features, and train `models/ver2/latest.pt` yourself. Version 1 checkpoints are rejected. Full-recording tonic estimation requires the whole supplied audio and can use substantial RAM; an incorrect estimate misaligns every window consistently.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/ver2/config.json").is_file(), "Launch from the repository root or notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ver2.config import load_config, resolve_path
from src.ver2.inference.predict import predict

config = load_config()


## Choose the audio and checkpoint

Edit these two paths. Relative audio paths are relative to the repository. The checkpoint supplies its own saved preprocessing and model settings; current config edits cannot alter the trained representation.

In [ ]:
AUDIO_PATH = ROOT / "your_audio.wav"
CHECKPOINT_PATH = resolve_path(config["paths"]["run"]) / "latest.pt"


In [ ]:
result = predict(AUDIO_PATH, CHECKPOINT_PATH)
print(result["message"])
print(f"Duration: {result['duration_seconds']:.3f} s | Device: {result['device']}")
print(f"Usable windows: {result['usable_windows']} | Failed windows: {result['failed_windows']}")

print("Full-recording Sa (Hz):", result["recording_tonic_hz"])


In [ ]:
import pandas as pd
from IPython.display import display

print("Overall ranking (real-duration-weighted window scores)")
display(pd.DataFrame(result["ranking"]))
columns = ["start_seconds", "valid_start_seconds", "valid_seconds", "left_padding_seconds",
           "right_padding_seconds", "tonic_hz", "top_raag", "failure"]
display(pd.DataFrame(result["windows"])[columns])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

successful = [row for row in result["windows"] if row["scores"] is not None]
if successful:
    times = [row["valid_start_seconds"] + row["valid_seconds"] / 2 for row in successful]
    scores = np.asarray([row["scores"] for row in successful])
    fig, ax = plt.subplots(figsize=(12, 4))
    for index, label in enumerate(result["classes"]):
        ax.plot(times, scores[:, index], marker=".", label=label)
    ax.set(xlabel="Window midpoint in recording (seconds)", ylabel="Comparative score",
           ylim=(0, 1), title="MEIKURAL — window predictions")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    fig.tight_layout()
    plt.show()
else:
    print("No score timeline: inspect window failures above.")


## Interpretation

The overall ranking averages successful windows, weighted by their real duration. It reduces the contribution of padded windows but does not make overlapping windows independent. A changing raag or key within a song can make the whole-recording average misleading; inspect the timeline.

No unknown-raag rejection is implemented. Tonic estimates can be wrong even when extraction succeeds. Very short clips and speech are especially unreliable. A decode error stops inference rather than returning a result from a truncated file. No external evaluation has been performed yet.